# 💳 Fraud Detection with Machine Learning

**Oasis Infobyte — Data Analytics — Level 1, Task 3**

This project detects fraudulent financial transactions in a heavily imbalanced dataset. It covers EDA, duplicate handling, SMOTE, stratified splitting, Logistic Regression, HistGradientBoosting, Precision, Recall, F1-score, ROC-AUC and scalability.

Expected dataset: `../data/creditcard.csv`. The raw CSV is not committed because it is larger than GitHub's 100 MB single-file limit.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
DATA_PATH = '../data/creditcard.csv'
if not Path(DATA_PATH).exists():
    raise FileNotFoundError('Place creditcard.csv in the data folder before running this notebook.')
df = pd.read_csv(DATA_PATH)
print('Original shape:', df.shape)
display(df.head())

## 1. Data inspection and class imbalance

The dataset contains 284,807 transactions and 31 columns. Fraud is extremely rare, so accuracy alone would be misleading. We focus on Precision, Recall, F1-score and ROC-AUC.

In [ ]:
print(df.info())
display(df.describe().T)
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))

# Remove duplicate transactions before modelling
df = df.drop_duplicates().copy()
print('Shape after removing duplicates:', df.shape)
class_counts = df['Class'].value_counts().sort_index()
print(class_counts)
print(f"Fraud rate: {df['Class'].mean()*100:.4f}%")

plt.figure(figsize=(7,4))
sns.countplot(data=df, x='Class')
plt.title('Legitimate vs Fraudulent Transactions')
plt.xlabel('Class (0 = Legitimate, 1 = Fraud)')
plt.show()

## 2. Exploratory Data Analysis

The data includes `Time` (seconds from the first transaction) and `Amount`. We inspect transaction amounts and an approximate hour-of-day pattern.

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(data=df, x='Amount', hue='Class', bins=80, element='step', stat='density', common_norm=False)
plt.xlim(0, df['Amount'].quantile(0.995))
plt.title('Transaction Amount: Fraud vs Legitimate')
plt.show()

df['Hour'] = ((df['Time'] % 86400) // 3600).astype(int)
hourly = df.groupby('Hour')['Class'].agg(['count','sum'])
hourly['fraud_rate_pct'] = hourly['sum'] / hourly['count'] * 100
plt.figure(figsize=(11,5))
sns.lineplot(data=hourly, x=hourly.index, y='fraud_rate_pct', marker='o')
plt.title('Approximate Fraud Rate by Hour')
plt.xlabel('Hour')
plt.ylabel('Fraud rate (%)')
plt.xticks(range(24))
plt.show()

## 3. Train/test split and SMOTE

Stratification ensures that the rare fraud class is represented in both sets. SMOTE is applied **only to the training data inside each pipeline**, preventing test-set leakage.

For computational practicality, SMOTE uses `sampling_strategy=0.2`, meaning the minority class is increased to 20% of the majority-class count rather than creating a 1:1 balance.

In [ ]:
X = df.drop(columns='Class')
y = df['Class']
features = X.columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Fraud in train:', int(y_train.sum()), '| Fraud in test:', int(y_test.sum()))

smote = SMOTE(sampling_strategy=0.2, random_state=RANDOM_STATE)
lr = Pipeline([('scaler', StandardScaler()), ('smote', smote), ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
hgb = Pipeline([('smote', SMOTE(sampling_strategy=0.2, random_state=RANDOM_STATE)), ('model', HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=31, learning_rate=0.08, random_state=RANDOM_STATE))])

lr.fit(X_train, y_train)
hgb.fit(X_train, y_train)
lr_pred, lr_prob = lr.predict(X_test), lr.predict_proba(X_test)[:,1]
hgb_pred, hgb_prob = hgb.predict(X_test), hgb.predict_proba(X_test)[:,1]

## 4. Model evaluation

Recall is important because missed fraud can be costly, while Precision matters because too many false alerts can burden customers and investigators. The final threshold should therefore reflect the business cost of each error.

In [ ]:
def metrics(name, y_true, pred, prob):
    return {'Model': name, 'Precision': precision_score(y_true,pred), 'Recall': recall_score(y_true,pred), 'F1-Score': f1_score(y_true,pred), 'ROC-AUC': roc_auc_score(y_true,prob)}

results = pd.DataFrame([metrics('Logistic Regression', y_test, lr_pred, lr_prob), metrics('HistGradientBoosting', y_test, hgb_pred, hgb_prob)])
display(results.style.format({c:'{:.4f}' for c in results.columns if c != 'Model'}))

print('Logistic Regression\n', classification_report(y_test, lr_pred, digits=4))
print('HistGradientBoosting\n', classification_report(y_test, hgb_pred, digits=4))

fig, ax = plt.subplots(1,2,figsize=(12,4))
sns.heatmap(confusion_matrix(y_test,lr_pred),annot=True,fmt='d',ax=ax[0])
ax[0].set_title('Logistic Regression')
sns.heatmap(confusion_matrix(y_test,hgb_pred),annot=True,fmt='d',ax=ax[1])
ax[1].set_title('HistGradientBoosting')
plt.tight_layout(); plt.show()

lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
hgb_fpr, hgb_tpr, _ = roc_curve(y_test, hgb_prob)
plt.figure(figsize=(8,6))
plt.plot(lr_fpr,lr_tpr,label=f'Logistic Regression AUC={roc_auc_score(y_test,lr_prob):.4f}')
plt.plot(hgb_fpr,hgb_tpr,label=f'HistGradientBoosting AUC={roc_auc_score(y_test,hgb_prob):.4f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC-AUC Curve'); plt.legend(); plt.show()

## 5. Completed results

The completed run produced the following test-set results after duplicate removal and stratified splitting:

| Model | Precision | Recall | F1-Score | ROC-AUC |
|---|---:|---:|---:|---:|
| Logistic Regression + SMOTE | 44.77% | 81.05% | 57.68% | 96.24% |
| HistGradientBoosting + SMOTE | **79.79%** | 78.95% | **79.37%** | **96.66%** |

HistGradientBoosting achieved the strongest overall balance of Precision, Recall, F1-score and ROC-AUC. Logistic Regression achieved slightly higher recall, which may be preferable when catching as many fraud cases as possible is the dominant business objective.

## 6. Scalability and conclusion

One million transactions per hour is approximately **278 transactions per second**. A production system could use streaming ingestion, precomputed behavioural features, parallel model-serving workers, batching where appropriate, threshold tuning, monitoring and periodic retraining.

### Conclusion
Fraud detection requires more than accuracy. This analysis removed duplicate rows, used stratification and leakage-safe SMOTE, and compared Logistic Regression with HistGradientBoosting using fraud-focused metrics. HistGradientBoosting was the strongest overall model in this completed run, while Logistic Regression offered higher recall. In deployment, the model and decision threshold should be selected using validation performance together with the financial cost of false positives and missed fraud.